# Brain Age Prediction: L'Analisi Definitiva (Multimodale)
Questo notebook esegue il test dei vari Ensemble (Simple e Avanzati) e, per la prima volta, implementa la **Age Bias Correction Statistica** fittata sul Validation Set originale del modello vincente per risolvere il problema della regressione verso la media (sovrastima dei giovani, sottostima degli anziani).

In [ ]:
!rm -rf SFCN
!git clone https://github.com/PietroSchgor/SFCN.git

import sys
sys.path.append('/kaggle/working/SFCN')

In [ ]:
import os
import json
import glob
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LinearRegression

from dp_model.model_files.sfcn import SFCN
from dp_model import dp_utils as dpu

PLOTS_DIR = '/kaggle/working/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

## 1. Dataset Custom (Multimodale)

In [ ]:
class BrainAgeDataset(Dataset):
    def __init__(self, data_dir, modality='FLAIR', is_train=False):
        self.data_dir = data_dir
        self.modality = modality 
        self.is_train = is_train
        self.subject_dirs = sorted(glob.glob(os.path.join(data_dir, "sub-*")))
        self.samples = []
        
        self.bin_range = [0, 70]
        self.bin_step = 1
        self.sigma = 1.0
        
        for subj_dir in self.subject_dirs:
            subj_id = os.path.basename(subj_dir)
            
            nii_path = os.path.join(subj_dir, f"{subj_id}_{self.modality}_MNI152_1mm.nii")
            if not os.path.exists(nii_path):
                nii_path = nii_path + ".gz"
                if not os.path.exists(nii_path):
                    if self.modality == 'T1w':
                        nii_path_alt = os.path.join(subj_dir, f"{subj_id}_T1_MNI152_1mm.nii.gz")
                        if os.path.exists(nii_path_alt):
                            nii_path = nii_path_alt
                        else:
                            continue
                    else:
                        continue
                    
            json_path = os.path.join(subj_dir, f"{subj_id}_participant_info.json")
            if not os.path.exists(json_path):
                continue
                
            with open(json_path, 'r') as f:
                info = json.load(f)
                
            participant_info = info.get("participant_info", {})
            age_cat_val = participant_info.get("age_scan")
            
            if age_cat_val is None:
                continue
            
            try:
                age_cat = int(age_cat_val) - 1
                true_age = 3 + age_cat * 5
                y, _ = dpu.num2vect(true_age, self.bin_range, self.bin_step, self.sigma)
            except:
                continue
            
            self.samples.append({
                "nii_path": nii_path,
                "label_vect": y,
                "true_age": true_age
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img = nib.load(sample['nii_path'])
        data = img.get_fdata(dtype=np.float32)
        
        mean_val = np.mean(data)
        if mean_val > 0:
            data = data / mean_val
            
        in_sp = data.shape
        out_sp = (160, 192, 160)
        
        dx, dy, dz = 0, 0, 0
        
        x_c = int((in_sp[0] - out_sp[0]) / 2) + dx
        y_c = int((in_sp[1] - out_sp[1]) / 2) + dy
        z_c = int((in_sp[2] - out_sp[2]) / 2) + dz
        
        data = data[x_c:x_c+out_sp[0], y_c:y_c+out_sp[1], z_c:z_c+out_sp[2]]
        data = np.expand_dims(data, axis=0)
        
        tensor_data = torch.from_numpy(data)
        label_vect = torch.tensor(sample['label_vect'], dtype=torch.float32)
        
        return tensor_data, label_vect, sample['true_age']

## 2. Configurazione Modelli e Estrazione Split (Train/Test/Val)

In [ ]:
KAGGLE_DATA_DIR = "/kaggle/input/datasets/elenaschgor/dataset-2-t1-flair/ds004199_final/"

PATHS_5_FOLD_FLAIR = [
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_1.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_2.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_3.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_4.pth",
    "/kaggle/input/sfcn-pesi/sfcn_FLAIR_fold_5.pth"
]
IDX_MIGLIOR_FLAIR = 0 

PATHS_5_FOLD_T1 = [
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_1.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_2.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_3.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_4.pth",
    "/kaggle/input/sfcn-pesi/sfcn_T1w_fold_5.pth"
]
IDX_MIGLIOR_T1 = 0 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dummy_dataset = BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False)
dataset_size = len(dummy_dataset)

if dataset_size > 0:
    all_ages = [sample['true_age'] for sample in dummy_dataset.samples]
    all_indices = np.arange(dataset_size)
    
    # SPLIT IDENTICO A QUELLO DI TRAINING (20% TEST, 80% TRAIN)
    train_idx, test_idx = train_test_split(all_indices, test_size=0.20, random_state=42, stratify=all_ages)
    
    test_dataset_flair = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False), test_idx)
    test_loader_flair = DataLoader(test_dataset_flair, batch_size=1, shuffle=False)
    
    test_dataset_t1 = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='T1w', is_train=False), test_idx)
    test_loader_t1 = DataLoader(test_dataset_t1, batch_size=1, shuffle=False)
    
    print("Caricamento delle 5 Reti FLAIR...")
    models_flair = []
    for p in PATHS_5_FOLD_FLAIR:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(p, map_location=device))
        m.to(device)
        m.eval()
        models_flair.append(m)
        
    print("Caricamento delle 5 Reti T1...")
    models_t1 = []
    for p in PATHS_5_FOLD_T1:
        m = SFCN(output_dim=70)
        m.load_state_dict(torch.load(p, map_location=device))
        m.to(device)
        m.eval()
        models_t1.append(m)

## 3. Calcolo Combinato e Generazione Plot Base

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print("   CALCOLO PREDITTIVO MULTIPLO SUL TEST SET")
    print("==============================================")
    
    all_true_ages = []
    
    preds_t1_best, preds_t1_ensemble = [], []
    preds_flair_best, preds_flair_ensemble = [], []
    preds_hybrid, preds_simple_hybrid = [], []
    
    bin_centers = np.arange(0, 70, 1)
    
    def process_batch(inputs_flair, inputs_t1):
        probs_flair_list = []
        for m in models_flair:
            out = m(inputs_flair)[0].view(1, -1)
            probs_flair_list.append(torch.exp(out).cpu().numpy())
            
        probs_t1_list = []
        for m in models_t1:
            out = m(inputs_t1)[0].view(1, -1)
            probs_t1_list.append(torch.exp(out).cpu().numpy())
            
        prob_flair_best = probs_flair_list[IDX_MIGLIOR_FLAIR]
        prob_flair_ensemble = np.mean(probs_flair_list, axis=0)
        
        prob_t1_best = probs_t1_list[IDX_MIGLIOR_T1]
        prob_t1_ensemble = np.mean(probs_t1_list, axis=0)
        
        prob_hybrid = (0.5 * prob_flair_best) + (0.5 * prob_t1_ensemble)
        prob_simple_hybrid = (0.5 * prob_flair_best) + (0.5 * prob_t1_best)
        
        ages = {
            'Best T1': (prob_t1_best @ bin_centers)[0],
            'Ens T1': (prob_t1_ensemble @ bin_centers)[0],
            'Best FLAIR': (prob_flair_best @ bin_centers)[0],
            'Ens FLAIR': (prob_flair_ensemble @ bin_centers)[0],
            'Hybrid Avanzato': (prob_hybrid @ bin_centers)[0],
            'Hybrid Semplice': (prob_simple_hybrid @ bin_centers)[0]
        }
        return ages

    with torch.no_grad():
        for (inputs_flair, _, true_age), (inputs_t1, _, _) in zip(test_loader_flair, test_loader_t1):
            inputs_flair, inputs_t1 = inputs_flair.to(device), inputs_t1.to(device)
            true_age_val = true_age.item()
            
            ages = process_batch(inputs_flair, inputs_t1)
            
            all_true_ages.append(true_age_val)
            preds_t1_best.append(ages['Best T1'])
            preds_t1_ensemble.append(ages['Ens T1'])
            preds_flair_best.append(ages['Best FLAIR'])
            preds_flair_ensemble.append(ages['Ens FLAIR'])
            preds_hybrid.append(ages['Hybrid Avanzato'])
            preds_simple_hybrid.append(ages['Hybrid Semplice'])
            
    mae_dict = {
        'Best T1': np.mean(np.abs(np.array(preds_t1_best) - all_true_ages)),
        'Ens T1': np.mean(np.abs(np.array(preds_t1_ensemble) - all_true_ages)),
        'Best FLAIR': np.mean(np.abs(np.array(preds_flair_best) - all_true_ages)),
        'Ens FLAIR': np.mean(np.abs(np.array(preds_flair_ensemble) - all_true_ages)),
        'Hybrid Avanzato': np.mean(np.abs(np.array(preds_hybrid) - all_true_ages)),
        'Hybrid Semplice': np.mean(np.abs(np.array(preds_simple_hybrid) - all_true_ages))
    }
    
    pred_dict = {
        'Best T1': preds_t1_best, 'Ens T1': preds_t1_ensemble,
        'Best FLAIR': preds_flair_best, 'Ens FLAIR': preds_flair_ensemble,
        'Hybrid Avanzato': preds_hybrid, 'Hybrid Semplice': preds_simple_hybrid
    }
    
    best_model_name = min(mae_dict, key=mae_dict.get)
    best_model_mae = mae_dict[best_model_name]
    winning_preds = np.array(pred_dict[best_model_name])
    all_true_ages = np.array(all_true_ages)
    
    print(f"\n>>> Il Modello Vincitore Assoluto è: {best_model_name} (MAE: {best_model_mae:.3f})")

## 4. Age Bias Correction Lineare (Fittata sul Validation Set Originale)
Per non compiere "Data Leakage" dal Test Set e non falsare la correzione fittandola sul Training Set (che la rete ha già memorizzato), ricreiamo i fold originari per isolare il **Validation Set** del modello vincitore, e fittiamo lì la Bias Correction.

In [ ]:
if dataset_size > 0:
    print("\n==============================================")
    print(f"  AGE BIAS CORRECTION SUL {best_model_name}")
    print("==============================================")
    
    # --- 1. RICREAZIONE STRATIFIED K-FOLD PER ESTRARRE I VAL_IDX --- 
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    train_val_ages = np.array(all_ages)[train_idx]
    
    # Scegliamo quale Fold usare come Validation Set (Se vince l'Hybrid, scegliamo quello FLAIR)
    target_fold = IDX_MIGLIOR_T1 if 'T1' in best_model_name and 'FLAIR' not in best_model_name else IDX_MIGLIOR_FLAIR
    
    val_idx_for_bias = []
    for fold, (f_train, f_val) in enumerate(skf.split(train_idx, train_val_ages)):
        if fold == target_fold:
            val_idx_for_bias = train_idx[f_val]
            break
            
    print(f"Estratti {len(val_idx_for_bias)} pazienti dal Validation Set originale del Fold {target_fold+1}.")
    
    # Creiamo i DataLoader ESCLUSIVAMENTE per il Validation Set estrattto
    val_dataset_flair = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='FLAIR', is_train=False), val_idx_for_bias)
    val_loader_flair = DataLoader(val_dataset_flair, batch_size=1, shuffle=False)
    
    val_dataset_t1 = torch.utils.data.Subset(BrainAgeDataset(KAGGLE_DATA_DIR, modality='T1w', is_train=False), val_idx_for_bias)
    val_loader_t1 = DataLoader(val_dataset_t1, batch_size=1, shuffle=False)
    
    val_true_ages = []
    val_winning_preds = []
    
    print("Calcolo predizioni sul Validation Set per fittare la retta...")
    with torch.no_grad():
        for (inputs_flair, _, true_age), (inputs_t1, _, _) in zip(val_loader_flair, val_loader_t1):
            inputs_flair, inputs_t1 = inputs_flair.to(device), inputs_t1.to(device)
            ages = process_batch(inputs_flair, inputs_t1)
            
            val_true_ages.append(true_age.item())
            val_winning_preds.append(ages[best_model_name])
            
    # --- 2. FIT DELLA REGRESSIONE LINEARE SUL VALIDATION SET ---
    X_val = np.array(val_winning_preds).reshape(-1, 1)
    Y_val = np.array(val_true_ages)
    
    bias_model = LinearRegression()
    bias_model.fit(X_val, Y_val)
    
    alpha = bias_model.coef_[0]
    beta = bias_model.intercept_
    print(f"\n>>> Formula Bias Correction trovata: Età_Corretta = {alpha:.3f} * Età_Predetta + {beta:.3f}")
    
    # --- 3. APPLICAZIONE DELLA CORREZIONE SUL TEST SET FINALE ---
    winning_preds_corrected = bias_model.predict(winning_preds.reshape(-1, 1))
    corrected_mae = np.mean(np.abs(winning_preds_corrected - all_true_ages))
    
    print(f"\n>>> MAE Prima della correzione : {best_model_mae:.3f} Anni")
    print(f">>> MAE DOPO LA BIAS CORRECTION: {corrected_mae:.3f} Anni <<<")
    
    # --- GRAFICO 1X2 CON ETICHETTE ESPLICITE ---
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
    
    # AX1: Bar Chart
    labels = list(mae_dict.keys())
    maes = list(mae_dict.values())
    colors = ['lightsteelblue', 'royalblue', 'lightcoral', 'crimson', 'gold', 'darkorange']
    bars = ax1.bar(labels, maes, color=colors, edgecolor='black', linewidth=1.5, alpha=0.9)
    for bar in bars:
        yval = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.1, 
                 f'{yval:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
    ax1.set_title('Confronto MAE Pre-Correzione', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Mean Absolute Error (Anni)')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', linestyle='--', alpha=0.5)
    ax1.set_ylim(0, max(maes) + 1.5)
    
    # AX2: Scatter Plot (Prima vs Dopo la Correzione)
    ax2.scatter(all_true_ages, winning_preds, color='silver', edgecolor='gray', alpha=0.6, s=50, label='Pre-Correzione')
    ax2.scatter(all_true_ages, winning_preds_corrected, color='forestgreen', edgecolor='black', alpha=0.9, s=90, marker='*', label='Post-Correzione')
    
    min_val = min(min(all_true_ages), min(winning_preds), min(winning_preds_corrected)) - 2
    max_val = max(max(all_true_ages), max(winning_preds), max(winning_preds_corrected)) + 2
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2.5, label='Ideale')
    
    # Titolo con i valori alpha e beta ben esposti
    ax2.set_title(f'Effetto Bias Correction sul {best_model_name}\nNuovo MAE: {corrected_mae:.3f} Anni\n(Formula: Età_Corretta = {alpha:.3f} * Età_Predetta + {beta:.3f})', 
                  fontsize=13, fontweight='bold', pad=15)
    ax2.set_xlabel('Età Reale (Anni)', fontsize=12)
    ax2.set_ylabel('Età Predetta (Anni)', fontsize=12)
    ax2.legend(loc='upper left')
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, '08_bias_corrected_results.png'), dpi=300)
    plt.show()
